# Sindhi chat model, 8B (Qwen3-8B), on Kaggle's free GPUs

Same recipe as the 4B on the laptop: cleaned Sindhi chat data (Apache-2.0 / CC-BY-SA sources), LoRA on a 4-bit model,
loss on answers only. Then the Sindhi reading test (Belebele, letter format) and answers to 20 held-out Sindhi requests.

**Settings before running** (right-hand panel):
1. *Accelerator*: **GPU T4 x2**
2. *Internet*: **On** (needs a phone-verified Kaggle account)

**Run it**: *Save Version* → **Save & Run All (Commit)**. It runs in the background (you can close the tab) and stops
itself after ~9 hours of training, leaving time to save and score inside Kaggle's 12-hour limit.

**Results**: open the finished version → *Output*: `results/belebele-mc-lora-chat-8b.json` (reading score),
`results/chat-8b.json` (answers), `out/lora-chat-8b/` (the trained model, ~0.7 GB).

**If it didn't finish one pass** (the last cell says so): *Add Input* → *Your Work* → this notebook's output, then
Save & Run All again. It continues from the last save.

In [ ]:
import os, subprocess
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.chdir("/kaggle/working")
!pip install -q -U transformers peft accelerate bitsandbytes huggingface_hub pyarrow
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
os.makedirs('sindhi', exist_ok=True)

In [ ]:
%%writefile sindhi/__init__.py



In [ ]:
%%writefile sindhi/evaluate.py
"""How well does a model handle Sindhi? Two numbers and some samples.
    python -m sindhi.evaluate                      # base model
    python -m sindhi.evaluate --adapter out/lora   # after training

perplexity:    on held-out Sindhi Wikipedia (data/test.txt). Lower is better; a model that has never seen
               Sindhi is surprised by every word.
bytes/token:   how many bytes of Sindhi text each token covers. Low means the tokenizer chops Sindhi into
               fragments (slow, expensive, and a sign the model never trained on it). Fixed by the vocabulary,
               not by LoRA, so it's reported once.
samples:       completions of a few Sindhi prompts, for a native speaker to judge. The model may well be
               fluent-sounding nonsense; perplexity can't tell you that, a Sindhi speaker can.
"""
from __future__ import annotations

import argparse
import json
import math
import time
from pathlib import Path

import torch

ROOT = Path(__file__).resolve().parents[1]
BASE = "Qwen/Qwen3-0.6B"
# 20 short Sindhi openers a native speaker can judge: history, places, people, everyday life
PROMPTS = ["سنڌ جي تاريخ", "ڪراچي شهر", "سنڌي ٻولي", "موهن جو دڙو", "شاهه عبداللطيف ڀٽائي", "سنڌو درياهه",
           "حيدرآباد سنڌ جو", "نوابشاهه", "سنڌي ثقافت", "اجرڪ ۽ ٽوپي", "سچل سرمست", "ٿر جو ريگستان", "سنڌ جي موسم",
           "سنڌ ۾ زراعت", "سنڌي ادب", "ڪراچي جو بندرگاهه", "سنڌ جو کاڌو", "لاڙڪاڻو", "سنڌ جي آبادي", "ميلا ۽ تهوار"]


def load(adapter: str | None, device: str, base: str = BASE, four_bit: bool = False):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(base)
    half = torch.bfloat16 if device == "cpu" or torch.cuda.is_bf16_supported() else torch.float16   # T4/P100: no bf16
    kw = {"dtype": half if device != "cpu" else torch.float32}
    if four_bit:   # same 4-bit loading as training, so base and trained are scored the same way
        from transformers import BitsAndBytesConfig
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                       bnb_4bit_compute_dtype=half,
                                                       bnb_4bit_use_double_quant=True)
        kw["device_map"] = {"": 0}
    model = AutoModelForCausalLM.from_pretrained(base, **kw)
    if adapter:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, adapter)
    return tok, (model if four_bit else model.to(device)).eval()


@torch.no_grad()
def perplexity(tok, model, lines: list[str], device: str, block: int = 512) -> tuple[float, float]:
    """(perplexity, bits per byte). Perplexity depends on the tokeniser; bits per byte doesn't, so it's the
    number to compare across models."""
    ids = tok("\n".join(lines), return_tensors="pt").input_ids[0]
    nll, n = 0.0, 0
    for i in range(0, len(ids) - 1, block):
        chunk = ids[i:i + block].unsqueeze(0).to(device)
        out = model(chunk, labels=chunk)          # the model shifts labels itself: next-token loss
        nll += out.loss.item() * (chunk.shape[1] - 1)
        n += chunk.shape[1] - 1
    nbytes = len("\n".join(lines).encode())
    return math.exp(nll / n), nll * (len(ids) - 1) / n / math.log(2) / nbytes


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--adapter")
    ap.add_argument("--lines", type=int, default=300, help="held-out paragraphs to score")
    ap.add_argument("--out", default="results")
    ap.add_argument("--base", default=BASE)
    ap.add_argument("--4bit", dest="four_bit", action="store_true")
    ap.add_argument("--name", help="results file name (default: base / the adapter folder name)")
    a = ap.parse_args()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tok, model = load(a.adapter, device, a.base, a.four_bit)
    lines = (ROOT / "data" / "test.txt").read_text(encoding="utf-8").splitlines()[:a.lines]
    text = "\n".join(lines)
    bpt = len(text.encode()) / len(tok(text).input_ids)
    t0 = time.time()
    ppl, bpb = perplexity(tok, model, lines, device)
    samples = {}
    for p in PROMPTS:
        ids = tok(p, return_tensors="pt").to(model.device)
        gen = model.generate(**ids, max_new_tokens=60, do_sample=True, temperature=0.7, top_p=0.9,
                             repetition_penalty=1.2, pad_token_id=tok.eos_token_id)
        samples[p] = tok.decode(gen[0][ids.input_ids.shape[1]:], skip_special_tokens=True)
    name = a.name or (Path(a.adapter).name if a.adapter else "base")
    res = {"model": a.base, "adapter": a.adapter, "perplexity": round(ppl, 2), "bits_per_byte": round(bpb, 4),
           "bytes_per_token": round(bpt, 2),
           "test_paragraphs": len(lines), "seconds": round(time.time() - t0), "samples": samples}
    out = ROOT / a.out
    out.mkdir(exist_ok=True)
    (out / f"{name}.json").write_text(json.dumps(res, ensure_ascii=False, indent=1), encoding="utf-8")
    print(json.dumps(res, ensure_ascii=True, indent=1))   # Windows consoles can't show Sindhi; the file has it


if __name__ == "__main__":
    main()


In [ ]:
%%writefile sindhi/belebele.py
"""Belebele snd_Arab (Meta, CC-BY-SA 4.0): 900 Sindhi reading-comprehension questions, four answers each.
Random guessing scores 25%. Scored zero-shot: the answer the model finds most likely (mean log-probability per
token, so long answers aren't penalised) after the passage and question.
--mc is the standard way to test a chat model instead: passage, question and the four options labelled A-D in a
chat message; the first letter the model writes is its answer (greedy, 8 tokens). --lang eng_Latn runs the English version,
which shows the ceiling: a model can't read Sindhi better than it reads English.
    python -m sindhi.belebele --base Qwen/Qwen3-4B --4bit [--adapter out/lora-4b] [--mc] [--lang eng_Latn]
"""
from __future__ import annotations

import argparse
import json
import re
import urllib.request
from pathlib import Path

import torch

from .evaluate import load

ROOT = Path(__file__).resolve().parents[1]
URL = "https://huggingface.co/datasets/facebook/belebele/resolve/main/data/{}.jsonl"
LETTERS = "ABCD"


@torch.no_grad()
def score(tok, model, context: str, answers: list[str]) -> int:
    ctx = tok(context, add_special_tokens=False).input_ids
    seqs = [ctx + tok(" " + a, add_special_tokens=False).input_ids for a in answers]
    width = max(map(len, seqs))
    pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    ids = torch.tensor([s + [pad] * (width - len(s)) for s in seqs], device=model.device)
    logits = model(ids, logits_to_keep=width - len(ctx) + 1).logits   # answer positions only: full vocab is huge
    best, best_i = -1e9, 0
    for i, s in enumerate(seqs):
        tgt = torch.tensor(s[len(ctx):], device=model.device)
        logp = torch.log_softmax(logits[i, :len(tgt)].float(), -1)
        lp = logp.gather(1, tgt[:, None]).mean().item()
        if lp > best:
            best, best_i = lp, i
    return best_i + 1


@torch.no_grad()
def score_mc(tok, model, it: dict) -> int:
    """The first A-D the model writes (chat models often open with "Answer:" or bold text), 0 if none."""
    opts = "\n".join(f"{L}) {it[f'mc_answer{i}']}" for i, L in enumerate(LETTERS, 1))
    msg = (f"{it['flores_passage']}\n\nQuestion: {it['question']}\n{opts}\n\n"
           "Answer with only the letter of the correct option (A, B, C or D).")
    prompt = tok.apply_chat_template([{"role": "user", "content": msg}], tokenize=False, add_generation_prompt=True,
                                     enable_thinking=False)
    ids = tok(prompt, add_special_tokens=False, return_tensors="pt").to(model.device)
    out = model.generate(**ids, max_new_tokens=8, do_sample=False, pad_token_id=tok.eos_token_id)
    m = re.search(r"\b([ABCD])\b", tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True))
    return LETTERS.index(m[1]) + 1 if m else 0


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--base", default="Qwen/Qwen3-4B")
    ap.add_argument("--adapter")
    ap.add_argument("--4bit", dest="four_bit", action="store_true")
    ap.add_argument("--limit", type=int, default=0)
    ap.add_argument("--name")
    ap.add_argument("--mc", action="store_true", help="options A-D in a chat prompt, answer = first letter written")
    ap.add_argument("--lang", default="snd_Arab")
    a = ap.parse_args()
    path = ROOT / "data" / ("belebele_snd.jsonl" if a.lang == "snd_Arab" else f"belebele_{a.lang}.jsonl")
    if not path.exists():
        urllib.request.urlretrieve(URL.format(a.lang), path)
    items = [json.loads(l) for l in path.read_text(encoding="utf-8").splitlines() if l.strip()]
    items = items[:a.limit] if a.limit else items
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tok, model = load(a.adapter, device, a.base, a.four_bit)
    right = 0
    for k, it in enumerate(items, 1):
        if a.mc:
            guess = score_mc(tok, model, it)
        else:
            ctx = f"{it['flores_passage']}\nسوال: {it['question']}\nجواب:"
            guess = score(tok, model, ctx, [it[f"mc_answer{i}"] for i in range(1, 5)])
        right += guess == int(it["correct_answer_num"])
        if k % 100 == 0:
            print(f"{k}/{len(items)}: {right / k:.1%}", flush=True)
    name = a.name or (Path(a.adapter).name if a.adapter else a.base.split("/")[-1])
    name = ("mc-" if a.mc else "") + (f"{a.lang}-" if a.lang != "snd_Arab" else "") + name
    res = {"model": a.base, "adapter": a.adapter, "method": "letter A-D" if a.mc else "answer likelihood",
           f"belebele_{a.lang}": round(right / len(items), 4), "n": len(items)}
    out = ROOT / "results"
    out.mkdir(exist_ok=True)
    (out / f"belebele-{name}.json").write_text(json.dumps(res, indent=1))
    print(json.dumps(res))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile sindhi/chat_data.py
"""Build the Sindhi chat training set: data/chat/train.jsonl ({"messages": [...], "source": ...}) and
data/chat/eval_prompts.json (20 held-out Sindhi requests for the A/B page). All sources Apache-2.0 or CC-BY-SA.

  MURI-IT snd          instructions written for real human Sindhi text (web, Wikipedia)       Apache-2.0
  Aya Dataset          274 prompts and answers written by native Sindhi speakers              Apache-2.0
  Sindhi-Assistant     1,500 conversations (Kashif786), instruction + cultural roleplay       Apache-2.0
  Aya Collection       machine-translated: Dolly (incl. answer-from-passage), chain-of-thought,
                       HotpotQA. Filtered hard: no English leaking in, no MT artefacts.         Apache-2.0
  Dolly-15k (English)  a little English so the model doesn't lose it                            CC-BY-SA-3.0

Nothing whose text overlaps Belebele's passages gets in, so the reading score stays honest.
    python -m sindhi.chat_data
"""
from __future__ import annotations

import json
import random
import re
from pathlib import Path

from huggingface_hub import HfFileSystem, hf_hub_download
import pyarrow.compute as pc
import pyarrow.parquet as pq

ROOT = Path(__file__).resolve().parents[1]
OUT = ROOT / "data" / "chat"
LATIN = re.compile(r"[A-Za-z]")
JUNK = re.compile(r"XNUMX|\?\?|ـ{2,}")
AYA_COLL = {"Dolly-v2 (T)": 6000, "Flan-CoT-submix (T)": 3000, "HotpotQA (T)": 1000}


def clean_sd(text: str, max_latin: float = 0.02) -> bool:
    """Sindhi text with almost no Latin letters (MT leaves English words behind) and no known MT artefacts."""
    return bool(text.strip()) and not JUNK.search(text) and len(LATIN.findall(text)) <= max_latin * len(text)


TAIL = re.compile(r"^\s*(حوالا|حوالو|حوالا جات|لاڳاپيل|ٻاهريان ڳنڍڻا|پڻ ڏسو|ٻاهريون ڳنڍڻا)\s*:?\s*$|"
                  r"^\s*(زمرو|زمرا|پيدائشون|وفاتون)\b", re.M)


def strip_wiki(text: str) -> str:
    """Wikipedia-sourced answers end with "حوالا" (References), "لاڳاپيل" (Related) and category lines; v3 learned to
    end every answer that way. Cut from the first such line, then drop empty section headings left at the end."""
    m = TAIL.search(text)
    lines = (text[:m.start()] if m else text).rstrip().split("\n")
    while m and len(lines) > 1 and (not lines[-1].strip() or (len(lines[-1].split()) <= 3 and lines[-1].strip()[-1:] not in ".؟!:")):
        lines.pop()
    return "\n".join(lines).strip()


def repetitive(text: str, n: int = 4) -> bool:
    """More than 10% of 4-word windows are repeats: machine-translation loops the model then copies."""
    w = text.split()
    grams = [" ".join(w[i:i + n]) for i in range(len(w) - n + 1)]
    return bool(grams) and (len(grams) - len(set(grams))) / len(grams) > 0.1


def pair(user: str, assistant: str, source: str) -> dict:
    return {"messages": [{"role": "user", "content": user.strip()}, {"role": "assistant", "content": assistant.strip()}],
            "source": source}


def belebele_shingles() -> set[str]:
    """8-word windows of every Belebele passage, to drop any training text that contains them."""
    path = ROOT / "data" / "belebele_snd.jsonl"
    if not path.exists():
        import urllib.request
        from .belebele import URL
        path.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve(URL.format("snd_Arab"), path)
    out = set()
    for line in path.read_text(encoding="utf-8").splitlines():
        w = json.loads(line)["flores_passage"].split()
        out.update(" ".join(w[i:i + 8]) for i in range(len(w) - 7))
    return out


def leaks(ex: dict, shingles: set[str]) -> bool:
    w = " ".join(m["content"] for m in ex["messages"]).split()
    return any(" ".join(w[i:i + 8]) in shingles for i in range(len(w) - 7))


def main() -> None:
    rng = random.Random(0)
    data: list[dict] = []
    get = lambda repo, f: hf_hub_download(repo, f, repo_type="dataset")  # noqa: E731

    muri = pq.read_table(get("akoksal/muri-it-language-split", "snd/train-00000-of-00001.parquet")).to_pylist()
    data += [pair(r["input"], r["output"], "MURI-IT") for r in muri
             if len(r["output"]) <= 1500 and clean_sd(r["input"]) and clean_sd(r["output"], 0.05)]

    aya = pq.read_table(get("CohereLabs/aya_dataset", "data/train-00000-of-00001.parquet"))
    aya = aya.filter(pc.equal(aya["language"], "Sindhi")).to_pylist()
    data += [pair(r["inputs"], r["targets"], "Aya Dataset") for r in aya]

    for line in Path(get("Kashif786/Sindhi-Assistant-IFT-1500", "data/all.jsonl")).read_text(encoding="utf-8").splitlines():
        msgs = [m for m in json.loads(line)["messages"] if m["role"] != "system"]
        if msgs and msgs[-1]["role"] == "assistant":
            data.append({"messages": msgs, "source": "Sindhi-Assistant-IFT-1500"})

    fs = HfFileSystem()
    coll = []
    for f in ("sindhi/train-00000-of-00002.parquet", "sindhi/train-00001-of-00002.parquet"):
        pf = pq.ParquetFile(fs.open(f"datasets/CohereLabs/aya_collection_language_split/{f}"))
        names = pf.read(columns=["dataset_name"]).column(0).to_pylist()
        groups, off = [], 0
        for g in range(pf.num_row_groups):
            n = pf.metadata.row_group(g).num_rows
            if set(names[off:off + n]) & AYA_COLL.keys():
                groups.append(g)
            off += n
        coll += [r for r in pf.read_row_groups(groups, columns=["inputs", "targets", "dataset_name"]).to_pylist()
                 if r["dataset_name"] in AYA_COLL]
    for name, k in AYA_COLL.items():
        rows = [r for r in coll if r["dataset_name"] == name and clean_sd(r["inputs"]) and clean_sd(r["targets"])]
        if name == "Dolly-v2 (T)":   # answer-from-a-passage rows first: that's the reading skill Belebele tests
            rows.sort(key=lambda r: "Context:" not in r["inputs"])
        else:
            rng.shuffle(rows)
        data += [pair(r["inputs"].replace(" / Context:", "\n\n").replace("Context:", "\n\n"), r["targets"],
                      f"Aya Collection {name}") for r in rows[:k]]

    dolly = [json.loads(l) for l in Path(get("databricks/databricks-dolly-15k", "databricks-dolly-15k.jsonl")).read_text().splitlines()]
    dolly = [r for r in dolly if r["category"] in ("closed_qa", "information_extraction", "open_qa", "classification")]
    rng.shuffle(dolly)
    data += [pair(r["instruction"] + ("\n\n" + r["context"] if r["context"] else ""), r["response"], "Dolly-15k (English)")
             for r in dolly[:1500]]

    shingles = belebele_shingles()
    kept = [ex for ex in data if not leaks(ex, shingles)]
    print(f"dropped {len(data) - len(kept)} examples that overlap Belebele passages")
    cut = 0
    for ex in kept:
        a = ex["messages"][-1]["content"]
        b = strip_wiki(a)
        cut += a != b
        ex["messages"][-1]["content"] = b
    before = len(kept)
    kept = [ex for ex in kept if ex["messages"][-1]["content"] and not repetitive(ex["messages"][-1]["content"])]
    print(f"cut Wikipedia endings from {cut} answers; dropped {before - len(kept)} repetitive or empty answers")

    val = pq.read_table(get("akoksal/muri-it-language-split", "snd/validation-00000-of-00001.parquet")).to_pylist()
    prompts = [r["input"] for r in val if len(r["input"]) <= 200 and clean_sd(r["input"])]
    rng.shuffle(prompts)

    rng.shuffle(kept)
    OUT.mkdir(parents=True, exist_ok=True)
    with (OUT / "train.jsonl").open("w", encoding="utf-8") as f:
        for ex in kept:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")
    (OUT / "eval_prompts.json").write_text(json.dumps(prompts[:20], ensure_ascii=False, indent=0), encoding="utf-8")
    counts: dict[str, int] = {}
    for ex in kept:
        counts[ex["source"]] = counts.get(ex["source"], 0) + 1
    print(json.dumps({"examples": len(kept), "by_source": counts}, ensure_ascii=False, indent=1))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile sindhi/chat_train.py
"""Chat tuning: teach the Sindhi-trained model to answer what it's asked. Continues the LoRA from the big
Sindhi run (out/lora-4b) on data/chat/train.jsonl, with the loss on the answer only (the model isn't trained to
write the question). Same habits as train_big: 4-bit, saves every 50 steps, stop any time, resumes.
    python -m sindhi.chat_train              # ~20k examples, one pass, ~4-5 h on an RTX 5070 Laptop
    python -m sindhi.chat_train --export     # newest save -> out/lora-chat-4b
On Kaggle (two T4s, no bf16), a fresh LoRA on the 8B that stops itself before the 12-hour session ends:
    accelerate launch --num_processes 2 -m sindhi.chat_train --base Qwen/Qwen3-8B --adapter none \
        --out out/ckpt-chat-8b --hours 9.5
"""
from __future__ import annotations

import argparse
import json
import os
import shutil
import sys
import time
from pathlib import Path

import torch

from .train_big import keep_awake, newest

ROOT = Path(__file__).resolve().parents[1]


def encode(tok, messages: list[dict], max_len: int) -> dict | None:
    """Prompt tokens masked with -100, answer tokens kept. None when the prompt alone leaves no room to answer."""
    prompt = tok.apply_chat_template(messages[:-1], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    p = tok(prompt, add_special_tokens=False).input_ids
    a = tok(messages[-1]["content"] + "<|im_end|>\n", add_special_tokens=False).input_ids
    if len(p) > max_len - 32:
        return None
    ids = (p + a)[:max_len]
    return {"input_ids": ids, "labels": [-100] * len(p) + ids[len(p):]}


class Chats(torch.utils.data.Dataset):
    def __init__(self, tok, max_len: int):
        rows = [json.loads(l) for l in (ROOT / "data" / "chat" / "train.jsonl").read_text(encoding="utf-8").splitlines()]
        self.items = [e for e in (encode(tok, r["messages"], max_len) for r in rows) if e]

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return {k: torch.tensor(v) for k, v in self.items[i].items()}


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--base", default="Qwen/Qwen3-4B")
    ap.add_argument("--adapter", default="out/lora-4b", help="the Sindhi LoRA to continue from, or 'none' for a fresh one")
    ap.add_argument("--rank", type=int, default=64, help="LoRA rank when --adapter none")
    ap.add_argument("--hours", type=float, default=0, help="stop (and save) after this long; 0 = no limit")
    ap.add_argument("--max-len", type=int, default=1024)
    ap.add_argument("--accum", type=int, default=16)
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--epochs", type=float, default=1.0)
    ap.add_argument("--save-steps", type=int, default=50)
    ap.add_argument("--max-steps", type=int, default=-1)
    ap.add_argument("--out", default="out/ckpt-chat-4b")
    ap.add_argument("--export", action="store_true")
    a = ap.parse_args()
    ckpt_dir = ROOT / a.out
    if a.export:
        c = newest(ckpt_dir)
        if not c:
            sys.exit("no checkpoint yet")
        dst = ckpt_dir.with_name(ckpt_dir.name.replace("ckpt", "lora"))   # out/ckpt-chat-4b -> out/lora-chat-4b
        dst.mkdir(parents=True, exist_ok=True)
        for f in ("adapter_config.json", "adapter_model.safetensors"):
            shutil.copy(c / f, dst / f)
        print(f"exported {c.name} -> {dst}")
        return

    from peft import LoraConfig, PeftModel, get_peft_model
    from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainerCallback, TrainingArguments

    class FreeBeforeSave(TrainerCallback):
        """Return cached GPU memory to the driver before each save: chat examples vary in length, so the cache
        grows, and on Windows a save on a nearly full 8 GB card froze the whole laptop (twice, at step 50)."""
        def on_step_end(self, args, state, control, **kw):
            # also every 10 steps: the cache crept past 8 GB over ~350 steps and steps slowed from 13 s to 110 s
            if (control.should_save or state.global_step % 10 == 0) and torch.cuda.is_available():
                torch.cuda.empty_cache()

    class TimeLimit(TrainerCallback):
        """Kaggle ends a session at 12 hours: stop in time to save, export and score. Constant LR, so any stop is usable."""
        start = time.time()

        def on_step_end(self, args, state, control, **kw):
            if not a.hours:
                return
            over = torch.tensor([float(time.time() - self.start > a.hours * 3600)],
                                device="cuda" if torch.cuda.is_available() else "cpu")
            if torch.distributed.is_available() and torch.distributed.is_initialized():
                torch.distributed.all_reduce(over, op=torch.distributed.ReduceOp.MAX)   # both GPUs stop on the same step
            if over.item():
                control.should_save = control.should_training_stop = True
    tok = AutoTokenizer.from_pretrained(a.base)
    ds = Chats(tok, a.max_len)
    cuda = torch.cuda.is_available()
    bf16 = cuda and torch.cuda.is_bf16_supported()   # Kaggle's T4/P100 have no bf16: fall back to fp16
    half = torch.bfloat16 if bf16 else torch.float16
    kw = {"dtype": half if cuda else torch.float32}
    if cuda:
        from transformers import BitsAndBytesConfig
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                       bnb_4bit_compute_dtype=half, bnb_4bit_use_double_quant=True)
        kw["device_map"] = {"": int(os.environ.get("LOCAL_RANK", 0))}   # one full copy per GPU under accelerate
    model = AutoModelForCausalLM.from_pretrained(a.base, **kw)
    if cuda:   # same memory settings as train_big (no fp32 upcast of the embedding)
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.enable_input_require_grads()
    if a.adapter == "none":
        model = get_peft_model(model, LoraConfig(
            r=a.rank, lora_alpha=a.rank, lora_dropout=0.0, task_type="CAUSAL_LM",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
    else:
        model = PeftModel.from_pretrained(model, str(ROOT / a.adapter), is_trainable=True)
    model.print_trainable_parameters()
    print(f"{len(ds):,} chat examples, {sum(len(x['input_ids']) for x in ds.items) / 1e6:.1f}M tokens per pass", flush=True)
    args = TrainingArguments(
        output_dir=str(ckpt_dir), per_device_train_batch_size=1, gradient_accumulation_steps=a.accum,
        learning_rate=a.lr, lr_scheduler_type="constant_with_warmup", warmup_steps=20, num_train_epochs=a.epochs,
        max_steps=a.max_steps, logging_steps=1, save_steps=a.save_steps, save_total_limit=3, bf16=bf16, fp16=cuda and not bf16,
        report_to=[], ddp_find_unused_parameters=False,
        # model + progress only: writing the paged optimizer's state froze Windows three times, always mid-file.
        # A resume restarts the optimizer (a brief wobble in the loss), never the progress.
        save_only_model=True,
        # paged: on an 8 GB card the non-paged optimizer spills into shared memory and runs 4x slower
        optim="paged_adamw_8bit" if cuda else "adamw_torch", seed=0, data_seed=0)
    keep_awake()
    resume = newest(ckpt_dir)
    if resume:
        print("resuming from", resume.name, flush=True)
    Trainer(model=model, args=args, train_dataset=ds, callbacks=[FreeBeforeSave(), TimeLimit()]).train(resume_from_checkpoint=str(resume) if resume else None)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile sindhi/chat_eval.py
"""Answers to 20 held-out Sindhi requests (data/chat/eval_prompts.json, never trained on), for the blind A/B page.
    python -m sindhi.chat_eval --4bit --name chat-base-4b                          # Qwen3-4B as released
    python -m sindhi.chat_eval --4bit --adapter out/lora-chat-4b --name chat-lora-chat-4b
    python -m sindhi.compare chat-base-4b chat-lora-chat-4b
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

import torch

from .evaluate import load

ROOT = Path(__file__).resolve().parents[1]


@torch.no_grad()
def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--base", default="Qwen/Qwen3-4B")
    ap.add_argument("--adapter")
    ap.add_argument("--4bit", dest="four_bit", action="store_true")
    ap.add_argument("--name", required=True)
    ap.add_argument("--max-new", type=int, default=220)
    ap.add_argument("--no-repeat", type=int, default=12,
                    help="never repeat a run of this many tokens (~4 Sindhi words); 0 = off")
    a = ap.parse_args()
    prompts = json.loads((ROOT / "data" / "chat" / "eval_prompts.json").read_text(encoding="utf-8"))
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tok, model = load(a.adapter, device, a.base, a.four_bit)
    torch.manual_seed(0)
    samples = {}
    for p in prompts:
        text = tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True,
                                       enable_thinking=False)
        ids = tok(text, add_special_tokens=False, return_tensors="pt").to(model.device)
        out = model.generate(**ids, max_new_tokens=a.max_new, do_sample=True, temperature=0.5, top_p=0.9,
                             repetition_penalty=1.1, no_repeat_ngram_size=a.no_repeat, pad_token_id=tok.eos_token_id)
        samples[p] = tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True).strip()
    out_dir = ROOT / "results"
    out_dir.mkdir(exist_ok=True)
    (out_dir / f"{a.name}.json").write_text(json.dumps({"model": a.base, "adapter": a.adapter, "samples": samples},
                                                       ensure_ascii=False, indent=1), encoding="utf-8")
    print(f"wrote results/{a.name}.json ({len(samples)} answers)")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile sindhi/train_big.py
"""QLoRA continued pre-training of Qwen3-4B on the big Sindhi corpus, built to run for many nights.

- Saves a checkpoint about every 30 minutes and resumes from the newest one on the next start: run it, stop it
  (close the window or Ctrl+C), run it again tomorrow, and it carries on.
- Constant learning rate after warm-up, so whichever checkpoint you stop at is a finished, usable adapter.
- Keeps Windows awake while it runs (it can't stop the laptop sleeping if the lid is shut with "sleep on lid").
    python -m sindhi.train_big                      # Qwen3-4B in 4-bit (needs bitsandbytes on the GPU)
    python -m sindhi.train_big --base Qwen/Qwen3-1.7B --no-4bit
    python -m sindhi.train_big --export             # copy the newest checkpoint's adapter to out/lora-4b
"""
from __future__ import annotations

import argparse
import json
import shutil
import sys
from pathlib import Path

import numpy as np
import torch

ROOT = Path(__file__).resolve().parents[1]
TOK = ROOT / "data" / "tokens"


class Blocks(torch.utils.data.Dataset):
    """Fixed-length blocks drawn from the token files: ~90% Sindhi, the rest Urdu and English, in one fixed
    shuffled order (the seed never changes, so a resumed run continues the same sequence)."""

    def __init__(self, block: int, seed: int = 0):
        meta = json.loads((TOK / "meta.json").read_text())
        self.block = block
        self.mm = {k: np.memmap(TOK / f"{k}.bin", dtype=np.uint32, mode="r") for k in ("sd", "ur", "en")
                   if (TOK / f"{k}.bin").exists()}
        idx = [(k, i * block) for k, m in self.mm.items() for i in range(len(m) // block - 1)]
        rng = np.random.default_rng(seed)
        order = rng.permutation(len(idx))
        self.idx = [idx[i] for i in order]
        self.base = meta["base"]

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        k, off = self.idx[i]
        x = torch.from_numpy(self.mm[k][off:off + self.block].astype(np.int64))
        return {"input_ids": x, "labels": x.clone()}


def keep_awake() -> None:
    if sys.platform == "win32":
        import ctypes
        ctypes.windll.kernel32.SetThreadExecutionState(0x80000000 | 0x00000001)   # ES_CONTINUOUS | ES_SYSTEM_REQUIRED


def newest(ckpt: Path) -> Path | None:
    # only complete saves: a save cut off by a power loss has no trainer_state.json and can't be resumed
    c = sorted((p.parent for p in ckpt.glob("checkpoint-*/trainer_state.json")), key=lambda p: int(p.name.split("-")[1]))
    return c[-1] if c else None


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--base", default="Qwen/Qwen3-4B")
    ap.add_argument("--no-4bit", action="store_true")
    ap.add_argument("--block", type=int, default=1024)
    ap.add_argument("--batch", type=int, default=2)
    ap.add_argument("--accum", type=int, default=16, help="2 x 16 x 1024 = 32k tokens per step")
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--rank", type=int, default=64)
    ap.add_argument("--save-steps", type=int, default=25, help="about every 30 min on an RTX 5070 Laptop (estimate)")
    ap.add_argument("--max-steps", type=int, default=0, help="0 = one pass over everything")
    ap.add_argument("--out", default="out/ckpt-4b")
    ap.add_argument("--export", action="store_true")
    a = ap.parse_args()
    ckpt_dir = ROOT / a.out
    if a.export:
        c = newest(ckpt_dir)
        if not c:
            sys.exit("no checkpoint yet")
        dst = ckpt_dir.with_name(ckpt_dir.name.replace("ckpt", "lora"))   # out/ckpt-4b -> out/lora-4b
        dst.mkdir(parents=True, exist_ok=True)
        for f in ("adapter_config.json", "adapter_model.safetensors"):
            shutil.copy(c / f, dst / f)
        print(f"exported {c.name} -> {dst}")
        return

    from peft import LoraConfig, get_peft_model
    from transformers import AutoModelForCausalLM, Trainer, TrainingArguments
    ds = Blocks(a.block)
    if ds.base != a.base:
        sys.exit(f"tokens were made with {ds.base}, not {a.base}: re-run sindhi.tokenize_corpus --base {a.base}")
    cuda = torch.cuda.is_available()
    kw = {"dtype": torch.bfloat16 if cuda else torch.float32}
    if cuda and not a.no_4bit:
        from transformers import BitsAndBytesConfig
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                       bnb_4bit_compute_dtype=torch.bfloat16,
                                                       bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(a.base, **kw)
    if cuda:
        # not peft's prepare_model_for_kbit_training: it upcasts the 389M-parameter tied embedding to fp32
        # (+1.5 GB), which pushes an 8 GB laptop card into shared system memory and ~20x slower steps
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.enable_input_require_grads()
        print(f"GPU memory free after loading: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB", flush=True)
    model = get_peft_model(model, LoraConfig(
        r=a.rank, lora_alpha=a.rank, lora_dropout=0.0, task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
    model.print_trainable_parameters()
    steps = a.max_steps or len(ds) // (a.batch * a.accum)
    print(f"{len(ds):,} blocks of {a.block} = {len(ds) * a.block / 1e6:.0f}M tokens; {steps:,} steps for one pass",
          flush=True)
    args = TrainingArguments(
        output_dir=str(ckpt_dir), per_device_train_batch_size=a.batch, gradient_accumulation_steps=a.accum,
        learning_rate=a.lr, lr_scheduler_type="constant_with_warmup", warmup_steps=20, max_steps=steps,
        logging_steps=1, save_steps=a.save_steps, save_total_limit=3, bf16=cuda, report_to=[],
        dataloader_pin_memory=cuda, optim="paged_adamw_8bit" if "quantization_config" in kw else "adamw_torch",
        seed=0, data_seed=0)
    keep_awake()
    resume = newest(ckpt_dir)
    if resume:
        print("resuming from", resume.name, flush=True)
    Trainer(model=model, args=args, train_dataset=ds).train(resume_from_checkpoint=str(resume) if resume else None)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile sindhi/compare.py
"""A blind A/B page for a Sindhi speaker: each prompt with the base and trained completions in random order.
Tap the better one; the page keeps score and shows the result. No typing.
    python -m sindhi.compare                       -> results/compare.html (base.json vs lora.json)
    python -m sindhi.compare base-4b lora-4b       -> results/compare-lora-4b.html
"""
import html
import json
import random
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
A, B = sys.argv[1:3] if len(sys.argv) > 2 else ("base", "lora")
base = json.loads((ROOT / "results" / f"{A}.json").read_text(encoding="utf-8"))["samples"]
lora = json.loads((ROOT / "results" / f"{B}.json").read_text(encoding="utf-8"))["samples"]
rng = random.Random(3)
items = []
for p in base:
    pair = [("base", base[p]), ("lora", lora[p])]
    rng.shuffle(pair)
    items.append({"prompt": p, "a": pair[0], "b": pair[1]})
page = """<!doctype html><html lang="sd" dir="rtl"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Sindhi A/B</title><style>body{font-family:'Noto Naskh Arabic','Segoe UI',sans-serif;background:#0b1724;color:#fff;margin:0;padding:20px;font-size:22px}
.q{margin:0 0 10px;color:#2aa6f5;font-size:28px}.c{display:block;width:100%;text-align:right;background:#132638;border:2px solid #2b4a66;color:#fff;border-radius:14px;padding:18px;margin:10px 0;font:inherit;line-height:1.7}
.c:active{border-color:#2aa6f5}#done{font-size:30px;text-align:center;margin-top:60px}small{color:#9fb3c6;font-size:16px;display:block;text-align:left;direction:ltr}</style>
<body><div id="app"></div><script>
const items=ITEMS;let i=0,score={base:0,lora:0,tie:0};const app=document.getElementById('app');
function show(){if(i>=items.length){const t=score.base+score.lora+score.tie;app.innerHTML=`<div id=done>Done.<br>trained ${score.lora} · base ${score.base} · same ${score.tie}<br><small>${t} prompts. Copy this line into results/human.txt</small></div>`;return}
const it=items[i];app.innerHTML=`<small>${i+1} / ${items.length}</small><p class=q>${it.prompt}…</p>
<button class=c onclick="pick('a')">${it.a[1]}</button><button class=c onclick="pick('b')">${it.b[1]}</button><button class=c onclick="pick('tie')" style="text-align:center;color:#9fb3c6">ٻئي ساڳيا / both as bad</button>`}
function pick(k){score[k==='tie'?'tie':items[i][k][0]]++;i++;show()}show();
</script></body></html>"""
out = ROOT / "results" / ("compare.html" if B == "lora" else f"compare-{B}.html")
out.write_text(page.replace("ITEMS", json.dumps(items, ensure_ascii=False)), encoding="utf-8")
print("open", out)


In [ ]:
# continue from a previous run's saves, if one was added as input
import glob, shutil
prev = sorted(glob.glob("/kaggle/input/*/out/ckpt-chat-8b/checkpoint-*/trainer_state.json"))
if prev and not os.path.exists("out/ckpt-chat-8b"):
    shutil.copytree(os.path.dirname(os.path.dirname(prev[0])), "out/ckpt-chat-8b")
    print("continuing from", sorted(os.listdir("out/ckpt-chat-8b")))
else:
    print("fresh start")

In [ ]:
!python -m sindhi.chat_data
!python -c "from huggingface_hub import snapshot_download; snapshot_download('Qwen/Qwen3-8B')"   # once, before 2 processes start

In [ ]:
import torch
n = torch.cuda.device_count()
launch = f"accelerate launch --num_processes {n} --mixed_precision fp16" if n > 1 else "python"
!{launch} -m sindhi.chat_train --base Qwen/Qwen3-8B --adapter none --out out/ckpt-chat-8b --hours 9 --save-steps 25

In [ ]:
!python -m sindhi.chat_train --export --out out/ckpt-chat-8b
!python -m sindhi.belebele --base Qwen/Qwen3-8B --4bit --mc --adapter out/lora-chat-8b --name lora-chat-8b
!python -m sindhi.chat_eval --base Qwen/Qwen3-8B --4bit --adapter out/lora-chat-8b --name chat-8b

In [ ]:
import json
state = json.load(open(sorted(glob.glob("out/ckpt-chat-8b/checkpoint-*/trainer_state.json"),
                              key=lambda p: int(p.split("checkpoint-")[1].split("/")[0]))[-1]))
print(f"trained {state['global_step']} of {state['max_steps']} steps"
      + ("" if state["global_step"] >= state["max_steps"] else " -> add this output as input and run again to finish"))
print(json.load(open("results/belebele-mc-lora-chat-8b.json")))
for q, ans in list(json.load(open("results/chat-8b.json", encoding="utf-8"))["samples"].items())[:3]:
    print("\nQ:", q, "\nA:", ans[:400])
# keep only the newest save so the output stays small
for c in sorted(glob.glob("out/ckpt-chat-8b/checkpoint-*"), key=lambda p: int(p.split("-")[-1]))[:-1]:
    shutil.rmtree(c)